# ScaNN streaming experiment (self-executable Colab)

Runs the same 6-cell streaming matrix the paper applies to IVF-PQ, but with **ScaNN** as the learned-codebook baseline. Goal: demonstrate that ScaNN's anisotropic AH codebook inherits PQ's staleness problem under streaming updates — moving the paper's claim about ScaNN from "inferred" to "demonstrated".

**Protocol.** For each of the 6 cells (2 datasets × 3 ScaNN memory regimes), run 1 seed first. If results look right, extend to 3 seeds (cell `RUN_MULTISEED = True` at the bottom). For each cell:
- Train on first 1M vectors (`scann.build()` with the cell's AH config).
- **Stale condition**: keep the searcher frozen; for new batches, store raw vectors in a flat searcher and at query time merge ScaNN top-K with flat top-K over the new vectors. This is the closest analog to IVF-PQ-stale.
- **Retrain condition**: rebuild ScaNN every batch on cumulative data. Cumulative compute is reported.
- Recall@10 measured against ground truth recomputed on the cumulative database every batch.

**Wall-clock budget** (Colab T4 GPU runtime; ScaNN is CPU-only but Colab CPUs are fast):
- Single seed × 6 cells × 10 batches × ScaNN-rebuild ~3 min/build = ~3 hours.
- Multi-seed (×3) = ~9 hours. Run in 2-3 sessions if needed.

**Output:** writes `scann_streaming_results.json` to the runtime's working directory. Download it at the end.

## 1. Setup

### 1a. Install (run once)

Colab ships Python 3.12 + NumPy 2.x. The previous pin to `scann==1.3.1` failed because that version was built for Python 3.10/3.11 + NumPy 1.x; the symptom was a `RecursionError` in `numpy/_dtype.py`. This cell installs scann's **latest** release (no version pin), which has Python 3.12 + NumPy 2.x wheels.

No kernel restart needed.

In [ ]:
# Colab on Python 3.12 + NumPy 2.x needs scann's latest release (1.4+),
# which has wheels for Python 3.10/3.11/3.12 and is built against NumPy 2.
# ScaNN 1.3.1 (Python 3.10/3.11 only, NumPy 1.x only) triggered the
# RecursionError in the previous attempt.
#
# This cell is idempotent: re-running it does nothing if scann is current.
# No kernel restart is forced.

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Uninstall any pinned-old scann that might be sitting around
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'scann'])

# Install the newest scann that matches the running Python + NumPy.
# Letting pip resolve `scann` (no version pin) picks the latest wheel
# compatible with this interpreter; on Python 3.12 that's scann>=1.4.0.
pip('scann', 'faiss-cpu', 'h5py')

print('scann install complete.')
print('Now run cell 1b below to import.')


### 1b. Imports

Should print `numpy: 2.x.x ... scann: 1.4.x` (or newer). If you get a `RecursionError`, the install in 1a didn't pick up scann's latest — re-run 1a.

In [ ]:
import json
import os
import time
from pathlib import Path
from importlib.metadata import version as _pkg_version

import numpy as np
import faiss
import scann

print('python  :', __import__('sys').version.split()[0])
print('numpy   :', np.__version__)
print('faiss   :', faiss.__version__)
print('scann   :', _pkg_version('scann'))


## 2. Dataset download

Downloads Deep-10M (~3.5 GB) into `./data/`. SIFT-10M is skipped on Colab because the only public mirror serves the full SIFT-1B file (~92 GB compressed). Deep-10M alone is sufficient for the ScaNN streaming-staleness evidence; the paper's Tables 5/6 already cover both datasets for IVF-PQ.

In [ ]:
DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

def download_if_missing(url, dest):
    dest = Path(dest)
    if dest.exists():
        sz = dest.stat().st_size / 1e9
        print(f'  exists: {dest} ({sz:.2f} GB)')
        return
    print(f'  downloading {url} → {dest}')
    !wget -q -O {dest} {url}

# Deep-10M (96-dim, image features) - ~3.5 GB
# Hosted by ann-benchmarks
download_if_missing(
    'http://ann-benchmarks.com/deep-image-96-angular.hdf5',
    DATA_DIR / 'deep10m.hdf5'
)

# NOTE: SIFT-10M skipped on Colab. The canonical bigann_base.bvecs.gz
# is the FULL SIFT-1B set (~92 GB compressed) — there's no pre-made
# 10M slice on the public mirror. Running only Deep-10M is sufficient
# evidence for the ScaNN streaming-staleness claim; the paper's main-text
# Tables 5 and 6 already cover both Deep-10M and SIFT-10M for IVF-PQ.

print('Done. Deep-10M only on Colab.')


In [ ]:
import h5py

def load_deep10m():
    """Load Deep-10M from ann-benchmarks hdf5 (already normalized)."""
    with h5py.File(DATA_DIR / 'deep10m.hdf5', 'r') as f:
        train = np.array(f['train'])  # ~10M × 96 already normalized
        test = np.array(f['test'])    # 10K queries
    return train.astype(np.float32), test.astype(np.float32)

# SIFT-10M loader removed for Colab to keep disk under 30 GB.


## 3. ScaNN streaming wrapper

ScaNN's pybind builder is static-build (no add method). We simulate streaming with two strategies:

**Stale-with-flat-extension** (closest to IVF-PQ-stale):
- Build ScaNN on initial 1M.
- For each new batch, store raw vectors in a flat IP searcher.
- At query time, search ScaNN over the original 1M *and* the flat searcher over new vectors, then merge top-K by score.
- This isolates the codebook-staleness problem: new vectors have full quality, but the ScaNN-side scoring uses the stale anisotropic codebook trained on the initial 1M.

**Retrain** (matches IVF-PQ-retrain):
- Rebuild ScaNN every batch on cumulative data.
- Cumulative build time reported.

In [ ]:
def build_scann(db, num_leaves, dims_per_block=2,
                aniso_threshold=0.2, training_sample_size=250_000,
                reorder_n=100):
    n, _ = db.shape
    builder = (
        scann.scann_ops_pybind.builder(db, 10, 'dot_product')
        .tree(num_leaves=num_leaves,
              num_leaves_to_search=max(1, num_leaves // 20),
              training_sample_size=min(training_sample_size, n))
        .score_ah(dimensions_per_block=dims_per_block,
                  anisotropic_quantization_threshold=aniso_threshold)
        .reorder(reorder_n)
    )
    return builder.build()

def search_with_stale_extension(searcher, scann_db, flat_extension, queries, k=10,
                                leaves_to_search=20):
    """Query ScaNN searcher over scann_db ∪ flat_extension.
    flat_extension is a 2D np.ndarray of raw vectors not in the ScaNN index.

    Memory-efficient: uses FAISS IndexFlatIP for the flat-extension search
    instead of materialising the full (Q × F) distance matrix in numpy,
    which would OOM at F=1M+ on standard runtimes.
    """
    n_scann = scann_db.shape[0]
    # ScaNN search
    s_idx, s_scores = searcher.search_batched(queries, leaves_to_search=leaves_to_search,
                                              pre_reorder_num_neighbors=100,
                                              final_num_neighbors=k * 2)
    # Flat search via FAISS (no full distance matrix materialised)
    if flat_extension is not None and len(flat_extension) > 0:
        flat_idx_local = faiss.IndexFlatIP(scann_db.shape[1])
        # Ensure contiguous float32 for FAISS
        flat_idx_local.add(np.ascontiguousarray(flat_extension, dtype=np.float32))
        f_top_scores, f_idx = flat_idx_local.search(
            np.ascontiguousarray(queries, dtype=np.float32), k * 2
        )
        # FAISS indices are 0-based into flat_extension; shift by n_scann
        combined_scores = np.concatenate([s_scores, f_top_scores], axis=1)
        combined_idx = np.concatenate([s_idx, f_idx + n_scann], axis=1)
        del flat_idx_local
    else:
        combined_scores = s_scores
        combined_idx = s_idx
    # Top k overall
    topk = np.argpartition(-combined_scores, kth=k - 1, axis=1)[:, :k]
    final_scores = np.take_along_axis(combined_scores, topk, axis=1)
    final_idx = np.take_along_axis(combined_idx, topk, axis=1)
    # Sort within top-k
    order = np.argsort(-final_scores, axis=1)
    return np.take_along_axis(final_scores, order, axis=1), np.take_along_axis(final_idx, order, axis=1)

def compute_recall_at_k(gt, pred, k=10):
    return np.mean([len(set(gt[i, :k]) & set(pred[i, :k])) for i in range(len(gt))]) / k


In [ ]:
# ScaNN memory regimes matched roughly to IVF-PQ sub/bit/super-matched.
# Memory ∝ num_leaves × dim_per_block (for the AH codebook size).
# These choices keep memory at the same axis as the paper's PQ regimes.
#
# Only Deep-10M is run on Colab (SIFT-10M dropped to fit 30GB disk).
CELLS = [
    # (dataset, regime_label, scann_kwargs)
    ('deep10m', 'sub-matched',   {'num_leaves': 2000, 'dims_per_block': 4}),
    ('deep10m', 'bit-matched',   {'num_leaves': 2000, 'dims_per_block': 2}),
    ('deep10m', 'super-matched', {'num_leaves': 4000, 'dims_per_block': 2}),
]

SEEDS_SINGLE = [42]
SEEDS_MULTI = [42, 123, 7777]
RUN_MULTISEED = False  # flip to True once single-seed pilot looks right

SEEDS = SEEDS_MULTI if RUN_MULTISEED else SEEDS_SINGLE

N_TRAIN_INIT = 1_000_000
N_TOTAL = 10_000_000
BATCH_SIZE = 1_000_000
N_QUERIES = 10_000
K = 10

datasets = {}
if any(c[0] == 'deep10m' for c in CELLS):
    print('Loading Deep-10M...')
    datasets['deep10m'] = load_deep10m()
print('Datasets loaded.')


In [ ]:
all_results = {}

for ds_name, regime, scann_kwargs in CELLS:
    base, queries = datasets[ds_name]
    base = base[:N_TOTAL]
    queries = queries[:N_QUERIES]
    dim = base.shape[1]
    # ann-benchmarks Deep-10M actually has ~9.99M rows, not 10M exactly.
    # Use the actual length and round the batch count to fit.
    n_actual = base.shape[0]
    n_batches = n_actual // BATCH_SIZE
    cell_key = f'{ds_name}_{regime}'
    print(f'\n=== CELL {cell_key} ({scann_kwargs}) ===')
    print(f'  base has {n_actual:,} vectors → {n_batches} batches of {BATCH_SIZE:,}')
    cell_results = {'config': {'dataset': ds_name, 'regime': regime,
                               'scann': scann_kwargs, 'dim': dim,
                               'n_actual': int(n_actual)}, 'seeds': {}}
    for seed in SEEDS:
        np.random.seed(seed)
        # Permute the base by seed (matches streaming_multiseed.py convention)
        perm = np.random.permutation(n_actual)
        b_perm = base[perm]
        init = b_perm[:N_TRAIN_INIT]
        # Build ScaNN stale on initial 1M
        t0 = time.time()
        scann_stale = build_scann(init, **scann_kwargs)
        build_stale_s = time.time() - t0
        print(f'  [seed={seed}] ScaNN-stale built on 1M in {build_stale_s:.1f}s')
        
        # Rebuild searcher will be re-instantiated every batch
        retrain_cum_s = 0.0
        
        records = []
        gt_index = faiss.IndexFlatIP(dim)
        gt_index.add(init)
        _, gt = gt_index.search(queries, k=K)
        
        stale_pred, _ = scann_stale.search_batched(queries, leaves_to_search=scann_kwargs['num_leaves']//20,
                                                    pre_reorder_num_neighbors=100, final_num_neighbors=K)
        retrain_pred = stale_pred  # at initial state, stale == retrain
        r_stale = compute_recall_at_k(gt, stale_pred, K)
        records.append({'step': 'init (1M)', 'n_indexed': N_TRAIN_INIT,
                        'scann_stale_r': r_stale, 'scann_retrain_r': r_stale,
                        'retrain_cum_s': 0.0})
        print(f'    init (1M): stale={r_stale:.1%}')
        
        # Stream batches
        for batch_idx in range(1, n_batches):
            new_start = batch_idx * BATCH_SIZE
            new_end = (batch_idx + 1) * BATCH_SIZE
            new_data = b_perm[new_start:new_end]
            cumulative = b_perm[:new_end]
            
            # Update flat extension for stale searcher
            flat_extension = b_perm[N_TRAIN_INIT:new_end] if new_end > N_TRAIN_INIT else None
            
            # Recompute GT against cumulative
            gt_index = faiss.IndexFlatIP(dim)
            gt_index.add(cumulative)
            _, gt = gt_index.search(queries, k=K)
            
            # Stale: ScaNN-1M + flat extension
            _, stale_idx = search_with_stale_extension(
                scann_stale, init, flat_extension, queries, k=K,
                leaves_to_search=scann_kwargs['num_leaves']//20
            )
            r_stale = compute_recall_at_k(gt, stale_idx, K)
            
            # Retrain: rebuild ScaNN on cumulative
            t0 = time.time()
            scann_retrain = build_scann(cumulative, **scann_kwargs)
            retrain_cum_s += time.time() - t0
            retrain_pred, _ = scann_retrain.search_batched(queries, leaves_to_search=scann_kwargs['num_leaves']//20,
                                                            pre_reorder_num_neighbors=100, final_num_neighbors=K)
            r_retrain = compute_recall_at_k(gt, retrain_pred, K)
            del scann_retrain
            
            records.append({
                'step': f'+{new_end//1_000_000}M',
                'n_indexed': new_end,
                'scann_stale_r': r_stale,
                'scann_retrain_r': r_retrain,
                'retrain_cum_s': round(retrain_cum_s, 1),
            })
            print(f'    {new_end//1_000_000}M: stale={r_stale:.1%} retrain={r_retrain:.1%} cum_compute={retrain_cum_s:.0f}s')
        
        cell_results['seeds'][str(seed)] = records
        all_results[cell_key] = cell_results
        # Save after every cell
        with open('scann_streaming_results.json', 'w') as f:
            json.dump(all_results, f, indent=2)
        print(f'  partial results saved.')
        del scann_stale

print('\n=== DONE ===')
print('Download scann_streaming_results.json from the file browser.')


## 5. Aggregation (paired-t across seeds)

Run this after `RUN_MULTISEED = True` has completed. Produces a summary table comparable to Tables 5–6 in the paper.

In [ ]:
from scipy.stats import t as student_t
from math import sqrt

def agg_cell(records_per_seed, metric):
    n_seeds = len(records_per_seed)
    n_steps = min(len(r) for r in records_per_seed)
    rows = []
    for s in range(n_steps):
        vals = [records_per_seed[i][s][metric] for i in range(n_seeds)]
        mn = float(np.mean(vals))
        sd = float(np.std(vals, ddof=1)) if n_seeds > 1 else 0.0
        t_crit = {2: 12.706, 3: 4.303, 4: 3.182}.get(n_seeds, 1.96)
        ci = t_crit * sd / sqrt(n_seeds) if n_seeds > 1 else 0.0
        rows.append({
            'step': records_per_seed[0][s]['step'],
            f'{metric}_mean': round(mn * 100, 3),
            f'{metric}_ci95': round(ci * 100, 3),
        })
    return rows

with open('scann_streaming_results.json') as f:
    all_results = json.load(f)

for cell_key, cell in all_results.items():
    print(f'\n=== {cell_key} ===')
    seeds_records = list(cell['seeds'].values())
    for metric in ['scann_stale_r', 'scann_retrain_r']:
        agg = agg_cell(seeds_records, metric)
        print(f'  {metric}:')
        for row in agg:
            print(f'    {row}')
    # Headline: change from 1M to 10M for stale and retrain
    final_stale = [r[-1]['scann_stale_r'] for r in seeds_records]
    initial_stale = [r[0]['scann_stale_r'] for r in seeds_records]
    diffs = [(f - i) * 100 for f, i in zip(final_stale, initial_stale)]
    n_seeds = len(diffs)
    mn = float(np.mean(diffs))
    sd = float(np.std(diffs, ddof=1)) if n_seeds > 1 else 0.0
    t_crit = {2: 12.706, 3: 4.303, 4: 3.182}.get(n_seeds, 1.96)
    ci = t_crit * sd / sqrt(n_seeds) if n_seeds > 1 else 0.0
    print(f'  Δ stale (1M → 10M): {mn:+.2f}pp ± {ci:.2f} (95% CI, paired across seeds)')